<a href="https://colab.research.google.com/github/rishabhrawat99/Internship-Projects/blob/main/ONLINE_RECOMENDATION_SYSTEM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_excel("/content/Online_Retail[1].xlsx")
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
user_product = df.pivot_table(
    index='CustomerID',
    columns='StockCode',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)
print("Matrix Shape:", user_product.shape)
similarity = cosine_similarity(user_product)
similarity_df = pd.DataFrame(
    similarity,
    index=user_product.index,
    columns=user_product.index
)
def recommend(user_id, n=5):
    if user_id not in user_product.index:
        return None
    similar_users = similarity_df.loc[user_id].sort_values(ascending=False)[1:6].index
    similar_data = user_product.loc[similar_users]
    scores = similar_data.mean().sort_values(ascending=False)
    user_items = user_product.loc[user_id]
    scores = scores[user_items == 0]
    if scores.empty:
        return None
    return scores.head(n)
user_input = input("Enter Customer ID: ")
try:
    user_input = int(float(user_input))
    recs = recommend(user_input, 5)
    if recs is None:
        print("No recommendations found or user does not exist.")
    else:
        print(f"\nRecommendations for User {user_input}:\n")
        for product, score in recs.items():
            product_rows = df[df['StockCode'] == product]
            if not product_rows.empty:
                name = product_rows['Description'].iloc[0]
                print(f"{name}  (Score: {round(score, 2)})")
            else:
                print(f"{product}  (Score: {round(score, 2)})")
except:
    print("Invalid input! Please enter a valid Customer ID.")

Matrix Shape: (4339, 3665)
Enter Customer ID: 17850

Recommendations for User 17850:

RED HANGING HEART T-LIGHT HOLDER  (Score: 20.0)
T-LIGHT HOLDER SWEETHEART HANGING  (Score: 11.2)
HANGING CLEAR MINI BOTTLE  (Score: 9.6)
DECORATIVE WICKER HEART SMALL  (Score: 9.6)
WORLD WAR 2 GLIDERS ASSTD DESIGNS  (Score: 9.6)
